# nDimensions.AI — Colab bootstrap

Run this at the top of every Colab session (Runtime → Change runtime type → GPU first).

**One-time setup** — add two secrets in the 🔑 *Secrets* panel (left sidebar) and enable notebook access:

| Secret | Value |
|---|---|
| `GH_TOKEN` | Your own fine-grained GitHub token, **read-only**, limited to `ndim-labs` repos |
| `WANDB_API_KEY` | From https://forge.coreweave.com/wandb/home |

Never paste tokens into cells or commit them.

In [ ]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")
# Colab disks are wiped on disconnect: checkpoints and datasets live on Drive.
WORK = "/content/drive/MyDrive/ndim-labs"
os.makedirs(f"{WORK}/checkpoints", exist_ok=True)
os.makedirs(f"{WORK}/data", exist_ok=True)

In [ ]:
token = userdata.get("GH_TOKEN")
REPO = "/content/ndim-infra"
if not os.path.exists(REPO):
    !git clone -q https://{token}@github.com/ndim-labs/ndim-infra.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}

In [ ]:
# Use Colab's preinstalled torch (matched to its CUDA); install only the rest.
!pip install -q tiktoken wandb datasets transformers peft accelerate bitsandbytes

In [ ]:
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_ENTITY"] = "ndim-labs"  # shared team; delete this line to log to your personal account
!python scripts/check_env.py

## Session rules
- Save a checkpoint to `WORK/checkpoints/<run-name>/` every ~15–30 min; Colab can disconnect without warning.
- Make every training script resumable from its latest checkpoint.
- Disconnect the runtime when you're done (Runtime → Disconnect and delete runtime). Compute units burn while it's idle.
- Log every run to W&B and write its entry in `research-log/`.